In [30]:
from dotenv import load_dotenv
load_dotenv()

True

In [31]:
from langchain.chat_models import init_chat_model
# llm=init_chat_model("groq:llama-3.3-70b-versatile")
llm = init_chat_model(
    "llama-3.1-8b-instant", 
    model_provider="groq"
)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000209FF6EC9D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000209FF6B7FD0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [39]:
from langchain_groq import ChatGroq
from deepeval.models.base_model import DeepEvalBaseLLM

class GroqModel(DeepEvalBaseLLM):

    def __init__(self):
        self.model = ChatGroq(
            # model="llama-3.3-70b-versatile",
            model="llama-3.1-8b-instant",
            temperature=0
        )

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        return self.model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        res = await self.model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return self.model.model_name
    

groq_judge = GroqModel()

print(
groq_judge.generate("Write me a joke")
)


A man walked into a library and asked the librarian, "Do you have any books on Pavlov's dogs and Schrödinger's cat?" 

The librarian replied, "It rings a bell, but I'm not sure if it's here or not."


In [40]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval

correctness_metric = GEval(
    name="Correctness",
    criteria="Determine if the 'actual output' is correct based on the 'expected output'.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    threshold=0.5,
    model=groq_judge
)

test_case = LLMTestCase(
    input="I have a persistent cough and fever. Should I be worried?",
    # Replace this with the actual output from your LLM application
    actual_output="A persistent cough and fever could signal various illnesses, from minor infections to more serious conditions like pneumonia or COVID-19. It's advisable to seek medical attention if symptoms worsen, persist beyond a few days, or if you experience difficulty breathing, chest pain, or other concerning signs.",
    expected_output="A persistent cough and fever could indicate a range of illnesses, from a mild viral infection to more serious conditions like pneumonia or COVID-19. You should seek medical attention if your symptoms worsen, persist for more than a few days, or are accompanied by difficulty breathing, chest pain, or other concerning signs."
)

evaluate([test_case], [correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using llama-3.1-8b-instant, strict=False, 
async_mode=True)...

c:\Users\Administrator\Downloads\demo\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                   ┃ Average Score                ┃ Pass Rate            ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Correctness [GEval]                      │ 0.80                         │ 100.00%              │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=5596107;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 0.88s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=True, score=0.8, reason="The Actual Output and Expected Output are mostly accurate, but there are some discrepancies in wording and phrasing. The Actual Output uses 'signal' instead of 'indicate', and the Expected Output uses 'mild viral infection' instead of 'minor infections'. Additionally, the Expected Output uses 'persist for more than a few days' instead of 'persist beyond a few days'.", strict_mode=False, evaluation_model='llama-3.1-8b-instant', error=None, evaluation_cost=None, verbose_logs='Criteria:\nDetermine if the \'actual output\' is correct based on the \'expected output\'. \n \nEvaluation Steps:\n[\n    "Compare the Actual Output with the Expected Output.",\n    "Determine if the Actual Output matches the Expected Output.",\n    "If the Actual Output matches the Expected Output, verify that all details are accurate.",\n  

In [41]:
from deepeval.test_case import LLMTestCase

test_cases = [

LLMTestCase(
    input="What is Kubernetes?",

    actual_output="""
    Kubernetes is a container orchestration platform.
    """,

    expected_output="""
    Kubernetes is an open-source container orchestration
    platform used to automate deployment, scaling,
    and management of containers.
    """
),

LLMTestCase(
    input="What is Docker?",

    actual_output="""
    Docker is used for containers.
    """,

    expected_output="""
    Docker is a platform used to build, package,
    and run applications inside containers.
    """
),

LLMTestCase(
    input="What is Terraform?",

    actual_output="""
    Terraform is Infrastructure as Code.
    """,

    expected_output="""
    Terraform is an Infrastructure as Code tool
    used to provision and manage cloud resources.
    """
)


]


In [42]:
evaluate(test_cases, [correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using llama-3.1-8b-instant, strict=False, 
async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What is Kubernetes?                                                                  │
│  │     Actual Output:                                                                                           │
│  │                             Kubernetes is a container orchestration platform.                                │
│  │                                                                                                              │
│  │     Expected Output:                                                                                         │
│  │                             Kubernetes is an open-source container orchestration                             │
│  │                             platform used to automate deployment, scaling,                                   │
│  │                             and management of containers.                                                    │
│  │                                                                                                              │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric              ┃ Score ┃ Threshold ┃ Reason                                                 │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Correctness [GEval] │ 0.20  │ 0.50      │ The Actual Output is missing several key details       │
│              │                     │       │           │ present in the Expected Output, such as                │
│              │                     │       │           │ 'open-source', 'automate deployment, scaling, and      │
│              │                     │       │           │ management of containers', which indicates a lack of   │
│              │                     │       │           │ accuracy in the Actual Output.                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              What is Docker?                                                                      │
│  │     Actual Output:                                                                                           │
│  │                             Docker is used for containers.                                                   │
│  │                                                                                                              │
│  │     Expected Output:                                                                                         │
│  │                             Docker is a platform used to build, package,                                     │
│  │                             and run applications insid

⚠ WARNING: No hyperparameters logged.
» ]8;id=5596109;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.96s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=False, score=0.2, reason="The Actual Output is missing several key details present in the Expected Output, such as 'open-source', 'automate deployment, scaling, and management of containers', which indicates a lack of accuracy in the Actual Output.", strict_mode=False, evaluation_model='llama-3.1-8b-instant', error=None, evaluation_cost=None, verbose_logs='Criteria:\nDetermine if the \'actual output\' is correct based on the \'expected output\'. \n \nEvaluation Steps:\n[\n    "Compare the Actual Output with the Expected Output.",\n    "Determine if the Actual Output matches the Expected Output.",\n    "If the Actual Output matches the Expected Output, verify that all details are accurate.",\n    "If the Actual Output does not match the Expected Output, identify the discrepancies and determine the cause."\n] \n \nRubric:\nNone \n \nScor

In [43]:
# Build DeepEval metrics JSON first. Inspect this before pushing to LangSmith.
import json
import time
from datetime import datetime, timezone
from pathlib import Path

from deepeval.test_case import LLMTestCase

if "correctness_metric" not in globals():
    raise RuntimeError("Run the earlier DeepEval cells first so correctness_metric is defined.")

# Add more DeepEval metrics here later, for example: [correctness_metric, faithfulness_metric, hallucination_metric]
deepeval_metrics = [correctness_metric]


def _clean_text(value):
    return value.strip() if isinstance(value, str) else value


def _metric_key(metric):
    return str(getattr(metric, "name", metric.__class__.__name__)).strip().replace(" ", "_").lower()


def _judge_model_name(metric):
    model = getattr(metric, "model", None)
    if model is not None and hasattr(model, "get_model_name"):
        return model.get_model_name()
    return str(getattr(metric, "evaluation_model", ""))


def _case_to_eval_kwargs(case: LLMTestCase) -> dict:
    kwargs = {
        "input": _clean_text(case.input),
        "actual_output": _clean_text(case.actual_output),
        "expected_output": _clean_text(case.expected_output),
    }
    for optional_field in ("context", "retrieval_context"):
        value = getattr(case, optional_field, None)
        if value is not None:
            kwargs[optional_field] = value
    return kwargs


deepeval_test_cases = []
if "test_case" in globals():
    deepeval_test_cases.append(test_case)
if "test_cases" in globals():
    deepeval_test_cases.extend(test_cases)
if not deepeval_test_cases:
    raise RuntimeError("Run the earlier cells first so test_case or test_cases is defined.")

run_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_UTC")
eval_output_dir = Path("eval_outputs")
eval_output_dir.mkdir(exist_ok=True)
deepeval_metrics_json_path = eval_output_dir / f"deepeval_metrics_{run_timestamp}.json"
langsmith_payload_json_path = eval_output_dir / f"langsmith_payload_{run_timestamp}.json"

# This is the JSON checkpoint you can inspect, save, or transform before LangSmith upload.
deepeval_metrics_json = []

for case in deepeval_test_cases:
    eval_case_kwargs = _case_to_eval_kwargs(case)
    row = {
        "question": eval_case_kwargs["input"],
        "answer": eval_case_kwargs["actual_output"],
        "expected_output": eval_case_kwargs.get("expected_output"),
        "latency": None,
        "model": None,
        "metrics": {},
        "metric_reasons": {},
        "metric_success": {},
    }

    started_at = time.perf_counter()
    for metric in deepeval_metrics:
        metric_name = _metric_key(metric)
        try:
            metric.measure(LLMTestCase(**eval_case_kwargs))
            row[metric_name] = float(metric.score) if metric.score is not None else None
            row["metrics"][metric_name] = row[metric_name]
            row["metric_reasons"][metric_name] = str(getattr(metric, "reason", ""))
            row["metric_success"][metric_name] = bool(getattr(metric, "success", False))
            row["model"] = row["model"] or _judge_model_name(metric)
        except Exception as exc:
            row[metric_name] = None
            row["metrics"][metric_name] = None
            row["metric_reasons"][metric_name] = f"DeepEval failed: {exc}"
            row["metric_success"][metric_name] = False
            row["model"] = row["model"] or _judge_model_name(metric)

    row["latency"] = round(time.perf_counter() - started_at, 3)
    deepeval_metrics_json.append(row)

# LangSmith-compatible payload drafted from the DeepEval JSON above.
langsmith_payload = {
    "dataset_name": f"DeepEval Groq Metrics {run_timestamp}",
    "examples": [
        {
            "inputs": {
                "question": item["question"],
                "actual_output": item["answer"],
                "deepeval_metrics": item["metrics"],
            },
            "outputs": {"expected_output": item.get("expected_output")},
            "metadata": {
                "source": "deepeval_metrics_json",
                "model": item.get("model"),
                "latency": item.get("latency"),
            },
        }
        for item in deepeval_metrics_json
    ],
    "feedback": [
        {
            "question": item["question"],
            "answer": item["answer"],
            "results": [
                {
                    "key": f"deepeval_{metric_name}",
                    "score": score,
                    "comment": item["metric_reasons"].get(metric_name),
                    "metadata": {
                        "success": item["metric_success"].get(metric_name),
                        "judge_model": item.get("model"),
                        "latency": item.get("latency"),
                    },
                }
                for metric_name, score in item["metrics"].items()
            ],
        }
        for item in deepeval_metrics_json
    ],
}

deepeval_metrics_json_path.write_text(json.dumps(deepeval_metrics_json, indent=2), encoding="utf-8")
langsmith_payload_json_path.write_text(json.dumps(langsmith_payload, indent=2), encoding="utf-8")

print(f"Saved DeepEval metrics JSON to: {deepeval_metrics_json_path.resolve()}")
print(f"Saved LangSmith payload JSON to: {langsmith_payload_json_path.resolve()}")
print(json.dumps(deepeval_metrics_json, indent=2))


[
  {
    "question": "I have a persistent cough and fever. Should I be worried?",
    "answer": "A persistent cough and fever could signal various illnesses, from minor infections to more serious conditions like pneumonia or COVID-19. It's advisable to seek medical attention if symptoms worsen, persist beyond a few days, or if you experience difficulty breathing, chest pain, or other concerning signs.",
    "expected_output": "A persistent cough and fever could indicate a range of illnesses, from a mild viral infection to more serious conditions like pneumonia or COVID-19. You should seek medical attention if your symptoms worsen, persist for more than a few days, or are accompanied by difficulty breathing, chest pain, or other concerning signs.",
    "latency": 0.748,
    "model": "llama-3.1-8b-instant",
    "metrics": {
      "correctness": 0.8
    },
    "metric_reasons": {
      "correctness": "The Actual Output and Expected Output are mostly accurate, but there are some discrepan

In [44]:
# Push the reviewed DeepEval JSON payload to LangSmith.
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from langsmith import Client

load_dotenv()
os.environ["LANGSMITH_TRACING"] = "true"

if "langsmith_payload" not in globals():
    payload_files = sorted(Path("eval_outputs").glob("langsmith_payload_*.json"))
    if not payload_files:
        raise RuntimeError("Run the previous cell first to create langsmith_payload JSON on disk.")
    langsmith_payload_json_path = payload_files[-1]
    langsmith_payload = json.loads(langsmith_payload_json_path.read_text(encoding="utf-8"))

if "deepeval_metrics_json" not in globals():
    metrics_files = sorted(Path("eval_outputs").glob("deepeval_metrics_*.json"))
    if not metrics_files:
        raise RuntimeError("Run the previous cell first to create deepeval_metrics JSON on disk.")
    deepeval_metrics_json_path = metrics_files[-1]
    deepeval_metrics_json = json.loads(deepeval_metrics_json_path.read_text(encoding="utf-8"))

client = Client()

dataset = client.create_dataset(
    dataset_name=langsmith_payload["dataset_name"],
    description="DeepEval JSON payload from 10_groq_deepeval.ipynb",
    metadata={"source_notebook": "10_groq_deepeval.ipynb"},
)
client.create_examples(
    dataset_id=dataset.id,
    examples=langsmith_payload["examples"],
)


def target(inputs: dict) -> dict:
    # Return the already-evaluated answer from the JSON payload.
    return {
        "answer": inputs["actual_output"],
        "deepeval_metrics": inputs.get("deepeval_metrics", {}),
    }


def feedback_from_deepeval_json(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    metrics = inputs.get("deepeval_metrics", {})
    matching_item = next(
        item for item in deepeval_metrics_json
        if item["question"] == inputs["question"] and item["answer"] == outputs["answer"]
    )

    return {
        "results": [
            {
                "key": f"deepeval_{metric_name}",
                "score": score,
                "comment": matching_item["metric_reasons"].get(metric_name),
                "metadata": {
                    "success": matching_item["metric_success"].get(metric_name),
                    "judge_model": matching_item.get("model"),
                    "latency": matching_item.get("latency"),
                    "source": "deepeval_metrics_json",
                },
            }
            for metric_name, score in metrics.items()
        ]
    }


experiment_results = client.evaluate(
    target,
    data=langsmith_payload["dataset_name"],
    evaluators=[feedback_from_deepeval_json],
    experiment_prefix="deepeval-json-groq",
    metadata={"source_notebook": "10_groq_deepeval.ipynb", "source_payload": "deepeval_metrics_json"},
    max_concurrency=1,
)

experiment_results.to_pandas()


View the evaluation results for experiment: 'deepeval-json-groq-b2a3cd1b' at:
https://smith.langchain.com/o/a54e226d-714c-4784-bf20-bbcd0a3e4974/datasets/e81a2f4f-db3d-47b1-8509-72538011e554/compare?selectedSessions=50b9b93f-30b1-4ebc-b464-034665a72cc3




4it [00:00, 57.14it/s]


,inputs.question,inputs.actual_output,inputs.deepeval_metrics,outputs.answer,outputs.deepeval_metrics,error,reference.expected_output,feedback.deepeval_correctness,execution_time,example_id,id
0,I have a persistent cough and fever. Should I ...,A persistent cough and fever could signal vari...,{'correctness': 0.8},A persistent cough and fever could signal vari...,{'correctness': 0.8},None,A persistent cough and fever could indicate a ...,0.8,0.0,889d60c4-7d0e-4506-a51f-321f34c096ef,019e7cde-93d7-7231-bbb3-cca9ea799c49
1,What is Docker?,Docker is used for containers.,{'correctness': 0.0},Docker is used for containers.,{'correctness': 0.0},None,"Docker is a platform used to build, package,\n...",0.0,0.0,a22b960b-2794-4ff7-9aae-b882b72d8851,019e7cde-93d7-7010-b044-78db4583606f
2,What is Terraform?,Terraform is Infrastructure as Code.,{'correctness': 0.2},Terraform is Infrastructure as Code.,{'correctness': 0.2},None,Terraform is an Infrastructure as Code tool\n ...,0.2,0.0,abecb14a-01b9-4643-9cb9-031c5ec42c09,019e7cde-93d7-7920-93b2-3ce37c1e1fd8
3,What is Kubernetes?,Kubernetes is a container orchestration platform.,{'correctness': 0.2},Kubernetes is a container orchestration platform.,{'correctness': 0.2},None,Kubernetes is an open-source container orchest...,0.2,0.0,ee9b528b-91e8-41a6-9662-cef9439173f2,019e7cde-93d7-7202-8772-a7fb2c5913cc
